# DoubleML DR-DiD — B1_strong_confounder

**Workstream B1 · canonical DiD**

large Var(alpha) and strong corr(alpha, treatment)

Doubly-robust DiD benchmark (DoubleML `DoubleMLIRM`, random-forest nuisances,
plugged into Callaway–Sant'Anna `att_gt`) for all linearity degrees 1/2/3 —
the same computation as `R_code/B1_strong_confounder_datasets/DoubleML_did.R`, moved to
Colab because it is the slow estimator. Panel: N=200, 4 pre + 4 post.

> **Colab:** upload just this notebook and *Run all*.

> ⚠️ **Runtime:** ~87s/iteration (B1) to ~129s/iteration (D) at `NUM_TREES=500`.
> Lower `REPS` for a quick check, or set `NUM_TREES=100` (≈ 4× faster; the forest
> is only a nuisance learner, so the doubly-robust estimate is ~unchanged).


In [ ]:
# ===== Parameters (edit me) =====
SCENARIO  = "B1_strong_confounder"
REPS      = 100     # replications per linearity degree (matches the DiD-BCF runs)
NUM_TREES = 500     # ranger trees for the DoubleML nuisances (100 ≈ 4x faster)
DATA_JOBS = 2       # parallel workers for the (cheap) data-generation step

# ===== Install R + the DoubleML stack as fast *binary* packages =====
import shutil, subprocess, os
if shutil.which('Rscript') is None:
    subprocess.run('sudo apt-get -qq update && sudo apt-get -qq install -y r-base',
                   shell=True, check=True)
codename = (subprocess.run(['bash','-lc','. /etc/os-release && echo $VERSION_CODENAME'],
            capture_output=True, text=True).stdout.strip() or 'jammy')
r_install = r'''
options(repos = c(CRAN = sprintf('https://packagemanager.posit.co/cran/__linux__/%s/latest', Sys.getenv('PPM_CODENAME'))),
        HTTPUserAgent = sprintf('R/%s R (%s)', getRversion(),
            paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
pkgs <- c('did','DoubleML','mlr3','mlr3learners','ranger','lgr','progress','openxlsx')
need <- pkgs[!pkgs %in% rownames(installed.packages())]
if (length(need)) install.packages(need)
cat('R packages ready:', paste(pkgs[pkgs %in% rownames(installed.packages())], collapse=', '), '\n')
'''
res = subprocess.run(['Rscript','-e', r_install], capture_output=True, text=True,
                     env={**os.environ, 'PPM_CODENAME': codename})
print(res.stdout[-3000:]); print(res.stderr[-1500:])


In [ ]:
# ===== Clone the engine + regenerate the seeded panels =====
import os, glob, subprocess
REPO_URL = "https://github.com/hugogobato/DiD-BCF.git"
if not os.path.isdir("DiD-BCF"):
    subprocess.run(["git","clone","--depth","1",REPO_URL], check=True)
ROOT = "DiD-BCF/Simulation_Studies_Revision"
%pip install -q numpy pandas joblib
# seed = replication index -> identical to the local panels
subprocess.run(f"cd {ROOT} && python DGPs/data_creation_{SCENARIO}.py --reps {REPS} --jobs {DATA_JOBS}", shell=True, check=True)

# Zero-pad iteration filenames so list.files() alphabetical order == numeric rep
# order (keeps the `iteration` column == replication number; no script edit needed).
folder = f"{ROOT}/R_code/{SCENARIO}_datasets"
for ld in glob.glob(f"{folder}/linearity_degree=*"):
    for fp in glob.glob(f"{ld}/iteration_*.csv"):
        n = int(os.path.basename(fp).split("_")[1].split(".")[0])
        new = os.path.join(ld, f"iteration_{n:04d}.csv")
        if fp != new:
            os.rename(fp, new)
print("panels ready (zero-padded) for", SCENARIO)


In [ ]:
# ===== Run DoubleML (loops linearity_degree=1/2/3) =====
if NUM_TREES != 500:
    subprocess.run(f"sed -i 's/num.trees = 500/num.trees = {NUM_TREES}/g' {folder}/DoubleML_did.R", shell=True, check=True)
subprocess.run(f"cd {folder} && Rscript DoubleML_did.R", shell=True, check=True)
print(open(f"{folder}/output_DoubleML_did.txt").read())


In [ ]:
# ===== Inspect + download the results (DiD-BCF-schema summaries) =====
import glob, sys, subprocess, pandas as pd
from IPython.display import display
sys.path.insert(0, ROOT)
from did_bcf_revision.metrics import compute_metrics, surface_metrics

csvs = sorted(glob.glob(f"{folder}/summaries_doubleml_{SCENARIO}_lin_*.csv"))
for f in csvs:
    print(f)
if csvs:
    summ = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)
    # Computed exactly like DiD-BCF (the engine's own functions). The CATT
    # surface uses the model's scalar estimate broadcast to each treated obs.
    print("\nDecomposed metrics (compute_metrics):")
    display(compute_metrics(summ))
    print("\nCATT-surface metrics (surface_metrics):")
    display(surface_metrics(summ))
else:
    print("No summaries written -- check the run-cell output above.")

zipname = "DoubleML_{}_results.zip".format(SCENARIO)
subprocess.run(f"cd {folder} && zip -q {zipname} summaries_doubleml_{SCENARIO}_lin_*.csv output_*.txt", shell=True)
try:
    from google.colab import files
    files.download(f"{folder}/{zipname}")
except Exception as e:
    print("(not on Colab / download skipped):", e)
